# Long-running Training + Asynchronous Evals

**The problem.** Real production training runs for hours or days. Real
evaluation batteries (held-out benchmarks, per-slice metrics, latency
profiling) take minutes to hours **per checkpoint**. You cannot afford
to inline them into the training loop, and you cannot afford to
serialise "train an epoch, then eval that checkpoint, then train the
next epoch".

**The anti-pattern.** Reopening the training run from an eval process
(`wandb.init(id=train_run_id, resume="allow")`) is unsafe — it causes
config overwriting, step interleaving, `run.finish()` races, and
summary last-write-wins. See
`notebooks/advanced/asynchronous_evals/ARCHITECTURE.md` for the
detailed treatment.

**The safe pattern.** *One job = one run*, tied together by a shared
`group=experiment_id` and by artifact lineage. This notebook
demonstrates it end-to-end on the workshop's TinyTransolver:

1. **§3 — Training run.** One `wandb.init(job_type="train")` owns
   its own run. Every 2 epochs it saves a checkpoint as a versioned
   model artifact whose metadata carries the identifiers eval
   consumers need.
2. **§4 — N async eval runs.** For each checkpoint, a *separate*
   `wandb.init(job_type="eval", group=..., resume="never")` opens
   its own run, `use_artifact`s the checkpoint (creating the lineage
   edge), runs the offline eval battery, and logs an eval-results
   artifact. In production each of these would fire on a different
   machine — here they run sequentially in one kernel; the API is
   identical either way.
3. **§5 — Aggregation.** After the eval runs finish, the API
   discovers them by server-side filter on
   `config.source_training_run_id`, computes best / latest, and
   writes compact `eval/*` keys back into the training run's
   summary via `api.run(...).summary.update({...})`. Namespaced
   under `eval/*` — nothing training-owned is overwritten.

**Prerequisites.** `registry.ipynb` executed so
`wandb-registry-airfoil_surrogates/datasets:production` exists in
`wandb_emea/jb_physics_demo`.


## 1. Setup

Third-party imports and constants. `experiment_id` names the group
that ties the training run and every eval run together in the W&B UI.


In [ ]:
from __future__ import annotations

import json
import math
import tempfile
import time
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb

device = torch.device("cpu")

ENTITY = "wandb_emea"
PROJECT = "jb_physics_demo"

REGISTRY_NAME = "airfoil_surrogates"
DATASET_COLLECTION = "datasets"
DATASET_REGISTRY_PATH = f"wandb-registry-{REGISTRY_NAME}/{DATASET_COLLECTION}:production"

experiment_id = f"exp-{uuid.uuid4().hex[:8]}"
group = experiment_id


def utc_ts() -> str:
    return datetime.now(timezone.utc).isoformat()


print(f"torch          = {torch.__version__}")
print(f"wandb          = {wandb.__version__}")
print(f"experiment_id  = {experiment_id}")
print(f"group          = {group}")


## 2. Model + data

Same `TinyTransolver` architecture as the rest of the workshop; the
`AirfoilCase` dataclass is used to typed-load the Registry-served
JSON. We compute normalisation stats once here — they are also
bundled into every checkpoint so eval jobs can rebuild the same
normalised inputs without needing the training run's state.


In [ ]:
class PhysicsAttention(nn.Module):
    def __init__(self, hidden: int, n_slices: int, n_heads: int, ffn_expansion: int = 2):
        super().__init__()
        assert hidden % n_heads == 0
        self.n_slices = n_slices
        self.n_heads = n_heads
        self.head_dim = hidden // n_heads

        self.slice_proj = nn.Linear(hidden, n_slices)
        self.qkv = nn.Linear(hidden, 3 * hidden)
        self.attn_out = nn.Linear(hidden, hidden)
        self.ffn = nn.Sequential(
            nn.Linear(hidden, ffn_expansion * hidden),
            nn.GELU(),
            nn.Linear(ffn_expansion * hidden, hidden),
        )
        self.norm1 = nn.LayerNorm(hidden)
        self.norm2 = nn.LayerNorm(hidden)

    def _slice_attend_deslice(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        M, H, D = self.n_slices, self.n_heads, self.head_dim
        w = torch.softmax(self.slice_proj(x), dim=-1)
        numer = torch.einsum("bnm,bnc->bmc", w, x)
        denom = w.sum(dim=1).unsqueeze(-1) + 1e-8
        z = numer / denom
        qkv = self.qkv(z).view(B, M, 3, H, D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(D)
        attn = torch.softmax(scores, dim=-1)
        z_out = (attn @ v).transpose(1, 2).contiguous().view(B, M, C)
        z_out = self.attn_out(z_out)
        return torch.einsum("bnm,bmc->bnc", w, z_out)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self._slice_attend_deslice(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class TinyTransolver(nn.Module):
    def __init__(
        self,
        in_dim: int = 5,
        out_dim: int = 3,
        hidden: int = 128,
        n_layers: int = 6,
        n_slices: int = 32,
        n_heads: int = 8,
        ffn_expansion: int = 2,
    ):
        super().__init__()
        self.embed = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([
            PhysicsAttention(hidden, n_slices, n_heads, ffn_expansion)
            for _ in range(n_layers)
        ])
        self.out_norm = nn.LayerNorm(hidden)
        self.head = nn.Linear(hidden, out_dim)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        x = self.embed(features)
        for block in self.blocks:
            x = block(x)
        return self.head(self.out_norm(x))


MODEL_CONFIG = dict(
    in_dim=5, out_dim=3, hidden=128, n_layers=6,
    n_slices=32, n_heads=8, ffn_expansion=2,
)


@dataclass
class AirfoilCase:
    points: np.ndarray
    sdf: np.ndarray
    aoa: float
    re: float
    pressure: np.ndarray
    velocity: np.ndarray

    def features(self) -> np.ndarray:
        n = self.points.shape[0]
        aoa_col = np.full((n, 1), self.aoa, dtype=np.float32)
        re_col = np.full((n, 1), np.log10(self.re) - 6.0, dtype=np.float32)
        return np.concatenate(
            [self.points.astype(np.float32),
             self.sdf[:, None].astype(np.float32),
             aoa_col, re_col],
            axis=1,
        )

    def targets(self) -> np.ndarray:
        return np.concatenate(
            [self.pressure[:, None], self.velocity], axis=1
        ).astype(np.float32)


# ---- Load the dataset from the Registry ----
api = wandb.Api()
ds_art = api.artifact(DATASET_REGISTRY_PATH)
ds_dir = Path(ds_art.download())
with open(next(ds_dir.glob("*.json"))) as f:
    payload = json.load(f)

cases = [
    AirfoilCase(
        points=np.array(c["points"], dtype=np.float32),
        sdf=np.array(c["sdf"], dtype=np.float32),
        aoa=float(c["aoa"]),
        re=float(c["re"]),
        pressure=np.array(c["pressure"], dtype=np.float32),
        velocity=np.array(c["velocity"], dtype=np.float32),
    )
    for c in payload["cases"]
]

n_val = int(len(cases) * 0.2)
train_cases, val_cases = cases[:-n_val], cases[-n_val:]

X_train = torch.stack([torch.from_numpy(c.features()).float() for c in train_cases])
Y_train = torch.stack([torch.from_numpy(c.targets()).float() for c in train_cases])
X_val = torch.stack([torch.from_numpy(c.features()).float() for c in val_cases])
Y_val = torch.stack([torch.from_numpy(c.targets()).float() for c in val_cases])

FEAT_MEAN = X_train.mean(dim=(0, 1), keepdim=True)
FEAT_STD = X_train.std(dim=(0, 1), keepdim=True).clamp(min=1e-6)
TGT_MEAN = Y_train.mean(dim=(0, 1), keepdim=True)
TGT_STD = Y_train.std(dim=(0, 1), keepdim=True).clamp(min=1e-6)

X_TRAIN_N = ((X_train - FEAT_MEAN) / FEAT_STD).to(device)
Y_TRAIN_N = ((Y_train - TGT_MEAN) / TGT_STD).to(device)
X_VAL_N = ((X_val - FEAT_MEAN) / FEAT_STD).to(device)
Y_VAL_N = ((Y_val - TGT_MEAN) / TGT_STD).to(device)

# Per-case AoA (deg) and Re — used for the eval bucketing in §4.
val_aoa_deg = torch.tensor(
    [math.degrees(c.aoa) for c in val_cases], dtype=torch.float32
)
val_re = torch.tensor([c.re for c in val_cases], dtype=torch.float32)

torch.manual_seed(0)
print(f"pulled {DATASET_REGISTRY_PATH}")
print(f"train: X={tuple(X_TRAIN_N.shape)} Y={tuple(Y_TRAIN_N.shape)}    "
      f"val: X={tuple(X_VAL_N.shape)} Y={tuple(Y_VAL_N.shape)}")


## 3. Training run — checkpoints as artifacts

The training job owns exactly one run (`job_type="train"`,
`resume="never"`). Every 2 epochs it saves a checkpoint file
containing the state dict, model config, and normalisation stats,
and logs it as a versioned `wandb.Artifact(type="model")`. The
artifact metadata carries the identifiers every eval consumer needs
(`experiment_id`, `group`, `training_run_id`, `epoch`, ...) — this
metadata dict is the contract between train and eval.

The training run also does a cheap in-line val step each epoch. That
is *not* the async eval battery — it's the fast loop-friendly signal
that shapes the training curve. Real evaluation is what §4 does.


In [ ]:
TRAIN_EPOCHS = 6
CHECKPOINT_EVERY = 2
BATCH_SIZE = 8
LR = 3e-3

checkpoint_refs: list[str] = []
train_metrics_by_epoch: dict[int, dict] = {}
tmp_dir = Path(tempfile.mkdtemp())

with wandb.init(
    entity=ENTITY, project=PROJECT,
    name=f"train-{experiment_id}",
    group=group,
    job_type="train",
    resume="never",
    config={
        "experiment_id": experiment_id,
        "group": group,
        "job_type": "train",
        "epochs": TRAIN_EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "checkpoint_every": CHECKPOINT_EVERY,
        "model_name": "TinyTransolver",
        "model_config": MODEL_CONFIG,
        "started_at": utc_ts(),
    },
    tags=["notebook", "advanced", "async-eval", experiment_id],
) as train_run:
    # Dataset lineage.
    train_run.use_artifact(DATASET_REGISTRY_PATH)
    training_run_id = train_run.id
    train_run.config.update(
        {"training_run_id": training_run_id}, allow_val_change=True
    )

    model = TinyTransolver(**MODEL_CONFIG).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=LR)
    global_step = 0
    best_p_rmse = float("inf")

    for epoch in range(1, TRAIN_EPOCHS + 1):
        model.train()
        perm = torch.randperm(X_TRAIN_N.size(0))
        last_loss = float("nan")
        for start in range(0, X_TRAIN_N.size(0), BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            xb, yb = X_TRAIN_N[idx], Y_TRAIN_N[idx]
            pred = model(xb)
            loss = F.mse_loss(pred, yb)
            optim.zero_grad()
            loss.backward()
            optim.step()
            global_step += 1
            last_loss = loss.item()
            train_run.log(
                {"train/loss": last_loss, "epoch": epoch},
                step=global_step,
            )

        # Cheap in-line val — separate from the async eval battery in §4.
        model.eval()
        with torch.no_grad():
            err = model(X_VAL_N) - Y_VAL_N
            p_rmse = err[..., 0].pow(2).mean().sqrt().item()
        best_p_rmse = min(best_p_rmse, p_rmse)
        train_metrics_by_epoch[epoch] = {
            "train/loss": last_loss,
            "val/pressure_rmse": p_rmse,
        }
        train_run.log(
            {"val/pressure_rmse": p_rmse, "epoch": epoch},
            step=global_step,
        )

        if epoch % CHECKPOINT_EVERY == 0 or epoch == TRAIN_EPOCHS:
            ckpt_path = tmp_dir / f"ckpt_ep{epoch:03d}.pt"
            torch.save(
                {
                    "state_dict": model.state_dict(),
                    "config": dict(MODEL_CONFIG),
                    "epoch": epoch,
                    "global_step": global_step,
                    "feat_mean": FEAT_MEAN.squeeze().tolist(),
                    "feat_std": FEAT_STD.squeeze().tolist(),
                    "tgt_mean": TGT_MEAN.squeeze().tolist(),
                    "tgt_std": TGT_STD.squeeze().tolist(),
                },
                ckpt_path,
            )
            art = wandb.Artifact(
                name=f"model-{experiment_id}",
                type="model",
                description=(
                    f"Checkpoint at epoch {epoch} for "
                    f"experiment {experiment_id}."
                ),
                metadata={
                    "experiment_id": experiment_id,
                    "group": group,
                    "training_run_id": training_run_id,
                    "epoch": epoch,
                    "global_step": global_step,
                    "model_name": "TinyTransolver",
                    "checkpoint_type": "epoch",
                    "created_by_job_type": "train",
                    "created_at": utc_ts(),
                    "train_metrics_at_checkpoint": train_metrics_by_epoch[epoch],
                },
            )
            art.add_file(str(ckpt_path), name="checkpoint.pt")
            logged = train_run.log_artifact(art)
            logged.wait()
            ref = (
                f"{train_run.entity}/{train_run.project}/"
                f"model-{experiment_id}:{logged.version}"
            )
            checkpoint_refs.append(ref)
            print(f"epoch {epoch:>2}: val/pressure_rmse={p_rmse:.4f}   "
                  f"checkpoint -> {ref}")
        else:
            print(f"epoch {epoch:>2}: val/pressure_rmse={p_rmse:.4f}")

    train_run.summary.update({
        "train/best_val_pressure_rmse": best_p_rmse,
        "train/total_epochs": TRAIN_EPOCHS,
        "train/total_steps": global_step,
        "train/checkpoint_count": len(checkpoint_refs),
        "train/checkpoint_refs": checkpoint_refs,
        "finished_at": utc_ts(),
    })

print()
print(f"experiment_id   = {experiment_id}")
print(f"training_run_id = {training_run_id}")
print(f"logged {len(checkpoint_refs)} checkpoints")


## 4. Async eval — one run per checkpoint

Each iteration below is the shape of a standalone eval script. In
production they would launch on separate machines, triggered by the
checkpoint artifact appearing on the backend (W&B Automations,
GitHub Actions, Argo, Slurm — anything). Here they run sequentially
inside one kernel; **the API is identical either way**.

Each eval run:

- Opens its own `wandb.init(job_type="eval", group=group,
  resume="never")` — never touches the training run.
- Reads the checkpoint's metadata to learn the epoch and model
  config.
- `use_artifact(...)` on the checkpoint to create the input lineage
  edge (eval run → checkpoint artifact).
- Runs the offline eval battery: normalised pressure/velocity RMSE,
  per-AoA-bucket RMSE, per-Re-bucket RMSE, inference latency.
- Logs a nested `eval-results` artifact with full metadata for
  auditability.
- Writes compact `eval/*` keys to its own summary.


In [ ]:
EVAL_BUCKETS_AOA = [(-8.0, 0.0), (0.0, 5.0), (5.0, 12.0)]
EVAL_BUCKETS_RE = [(1e6, 3e6), (3e6, 6e6), (6e6, 1e7 + 1)]

for checkpoint_ref in checkpoint_refs:
    api = wandb.Api()
    ckpt_art_info = api.artifact(checkpoint_ref, type="model")
    meta = ckpt_art_info.metadata or {}
    ep = int(meta["epoch"])

    with wandb.init(
        entity=ENTITY, project=PROJECT,
        name=f"eval-{experiment_id}-ep{ep:03d}",
        group=group,
        job_type="eval",
        resume="never",
        config={
            "experiment_id": experiment_id,
            "group": group,
            "job_type": "eval",
            "source_training_run_id": training_run_id,
            "source_checkpoint_artifact": checkpoint_ref,
            "source_checkpoint_version": ckpt_art_info.version,
            "source_checkpoint_epoch": ep,
            "eval_dataset": "synthetic_airfoil_slice",
            "eval_split": "val",
            "eval_config": {
                "eval_split": "val",
                "batch_size_eval": 16,
                "aoa_buckets": EVAL_BUCKETS_AOA,
                "re_buckets": EVAL_BUCKETS_RE,
                "eval_code_version": "v1.0.0",
            },
            "eval_timestamp": utc_ts(),
        },
        tags=["notebook", "advanced", "async-eval", experiment_id, "eval"],
    ) as eval_run:
        # eval_run --uses--> checkpoint_artifact (input lineage edge)
        
        input_art = eval_run.use_artifact(checkpoint_ref, type="model")
        
        d = Path(input_art.download())
        ckpt = torch.load(next(d.glob("*.pt")), map_location="cpu")

        eval_model = TinyTransolver(**ckpt["config"]).to(device)
        eval_model.load_state_dict(ckpt["state_dict"])
        eval_model.eval()

        t0 = time.perf_counter()
        with torch.no_grad():
            pred_n = eval_model(X_VAL_N)
        latency_ms = (time.perf_counter() - t0) * 1000.0
        err = pred_n - Y_VAL_N
        p_rmse = err[..., 0].pow(2).mean().sqrt().item()
        u_rmse = err[..., 1].pow(2).mean().sqrt().item()
        v_rmse = err[..., 2].pow(2).mean().sqrt().item()
        vel_rmse = float(((u_rmse ** 2 + v_rmse ** 2) / 2) ** 0.5)

        per_aoa: dict[str, float] = {}
        for lo, hi in EVAL_BUCKETS_AOA:
            mask = (val_aoa_deg >= lo) & (val_aoa_deg < hi)
            if mask.any():
                key = f"aoa_{int(lo)}_{int(hi)}"
                per_aoa[key] = err[mask, :, 0].pow(2).mean().sqrt().item()

        per_re: dict[str, float] = {}
        for lo, hi in EVAL_BUCKETS_RE:
            mask = (val_re >= lo) & (val_re < hi)
            if mask.any():
                key = f"re_{lo:.0e}_{hi:.0e}"
                per_re[key] = err[mask, :, 0].pow(2).mean().sqrt().item()

        # Chart-friendly flat scalars.
        flat = {
            "eval/pressure_rmse": p_rmse,
            "eval/velocity_rmse": vel_rmse,
            "eval/latency_ms_per_case": latency_ms / X_VAL_N.size(0),
            "eval/checkpoint_epoch": ep,
            "epoch": ep,
        }
        
        for k, v in per_aoa.items():
            flat[f"eval/{k}_pressure_rmse"] = v
        for k, v in per_re.items():
            flat[f"eval/{k}_pressure_rmse"] = v
        
        eval_run.log(flat)

        # Namespaced summary (used by the aggregator in §5).
        eval_run.summary.update({
            "eval/pressure_rmse": p_rmse,
            "eval/velocity_rmse": vel_rmse,
            "eval/checkpoint_epoch": ep,
            "eval/checkpoint_artifact": checkpoint_ref,
            "eval/split": "val",
            "eval/latency_ms_per_case": latency_ms / X_VAL_N.size(0),
        })

        # Full nested eval-results artifact for auditability.
        results_path = tmp_dir / f"eval_ep{ep:03d}.json"
        
        results_path.write_text(json.dumps({
            "experiment_id": experiment_id,
            "epoch": ep,
            "eval_run_id": eval_run.id,
            "source_training_run_id": training_run_id,
            "checkpoint_artifact": checkpoint_ref,
            "metrics": {
                "pressure_rmse": p_rmse,
                "velocity_rmse": vel_rmse,
                "latency_ms": latency_ms,
                "per_aoa": per_aoa,
                "per_re": per_re,
            },
            "evaluated_at": utc_ts(),
        }, indent=2))
        
        results_art = wandb.Artifact(
            name=f"eval-results-{experiment_id}",
            type="eval-results",
            metadata={
                "experiment_id": experiment_id,
                "group": group,
                "training_run_id": training_run_id,
                "source_training_run_id": training_run_id,
                "eval_run_id": eval_run.id,
                "checkpoint_artifact": checkpoint_ref,
                "source_checkpoint_artifact": checkpoint_ref,
                "checkpoint_epoch": ep,
                "source_checkpoint_epoch": ep,
                "eval_dataset": "synthetic_airfoil_slice",
                "eval_split": "val",
                "metrics_schema_version": "1.0",
                "created_at": utc_ts(),
            },
        )
        results_art.add_file(str(results_path), name="eval_metrics.json")
        eval_run.log_artifact(results_art)

        print(
            f"eval ep{ep}: pressure_rmse={p_rmse:.4f}   "
            f"velocity_rmse={vel_rmse:.4f}   "
            f"latency={latency_ms:.1f} ms   "
            f"run_id={eval_run.id}"
        )


## 5. Aggregate — merge eval results into the training run summary

Post-hoc aggregation. Uses `wandb.Api().runs(...)` with a server-side
filter on `config.source_training_run_id` to find every eval run tied
to this training run — no external state, no directory scanning.
Computes best/latest and writes compact keys back to the training run
via `api.run(...).summary.update({...})`. All keys are namespaced
under `eval/*` — nothing training-owned is touched.


In [ ]:
api = wandb.Api()
eval_runs = list(api.runs(
    f"{ENTITY}/{PROJECT}",
    filters={
        "config.source_training_run_id": training_run_id,
        "config.job_type": "eval",
    },
))
print(f"found {len(eval_runs)} eval run(s) for training_run_id={training_run_id}")

records: list[dict] = []
for r in eval_runs:
    s = dict(r.summary)
    c = dict(r.config)
    p_rmse = s.get("eval/pressure_rmse")
    ep = s.get("eval/checkpoint_epoch") or c.get("source_checkpoint_epoch")
    if p_rmse is None or ep is None:
        print(f"  [skip] {r.id} — missing eval/pressure_rmse or epoch")
        continue
    records.append({
        "run_id": r.id,
        "epoch": int(ep),
        "pressure_rmse": float(p_rmse),
        "velocity_rmse": float(s.get("eval/velocity_rmse", float("nan"))),
        "checkpoint_artifact": s.get("eval/checkpoint_artifact")
                                or c.get("source_checkpoint_artifact"),
    })

print()
print("eval records (sorted by epoch):")
for r in sorted(records, key=lambda x: x["epoch"]):
    print(f"  epoch {r['epoch']:>3}: "
          f"pressure_rmse={r['pressure_rmse']:.4f}   "
          f"run_id={r['run_id']}")

best = min(records, key=lambda r: r["pressure_rmse"])
latest = max(records, key=lambda r: r["epoch"])

agg = {
    "eval/best/pressure_rmse": best["pressure_rmse"],
    "eval/best/velocity_rmse": best["velocity_rmse"],
    "eval/best/checkpoint_epoch": best["epoch"],
    "eval/best/checkpoint_artifact": best["checkpoint_artifact"],
    "eval/best/run_id": best["run_id"],
    "eval/latest/pressure_rmse": latest["pressure_rmse"],
    "eval/latest/checkpoint_epoch": latest["epoch"],
    "eval/latest/checkpoint_artifact": latest["checkpoint_artifact"],
    "eval/latest/run_id": latest["run_id"],
    "eval/history_count": len(records),
    "eval/aggregated_at": utc_ts(),
    "eval/run_ids": [r["run_id"] for r in records],
}

print()

print("computed aggregate summary:")

for k, v in agg.items():
    if not isinstance(v, list):
        print(f"  {k}: {v}")

training_run = api.run(f"{ENTITY}/{PROJECT}/{training_run_id}")
training_run.summary.update(agg)
training_run.update()


print()
print(f"wrote {len(agg)} keys under eval/* into training run {training_run_id}")


## 6. Why not resume?

Reopening the training run from the eval process
(`wandb.init(id=train_run_id, resume="allow")`) sounds convenient
but breaks the run's data model:

- **Config overwrites.** Any config key the eval process writes
  merges silently on the server. If eval and training share a key
  by accident, the training value is lost — no conflict detection.
- **Step interleaving.** `run.log()` writes are append-only per
  run. Concurrent writers produce interleaved, non-monotonic step
  sequences; the resulting charts are meaningless.
- **`run.finish()` races.** Whichever process calls `finish()`
  first marks the run as finished. Subsequent logs from the other
  process go into a "finished" run.
- **Summary last-write-wins.** `run.summary` is a flat dict.
  Concurrent writes to different keys are OK in principle but
  order-dependent in practice.

None of these show up in a demo where you run everything in one
kernel — they show up in production when a training job takes 12
hours and eval workers spawn on a schedule. The multi-run + group
pattern above avoids every one of them structurally: each writer
owns its own run.

Long-form treatment:
`notebooks/advanced/asynchronous_evals/ARCHITECTURE.md`.


## 7. Production caveats

- **What "async" means outside the notebook.** Each eval-run cell
  above is the shape of a standalone script. In production spawn N
  of them on separate machines, triggered by the checkpoint
  artifact appearing (W&B Automations, GitHub Actions, Argo,
  Slurm — anything). They don't coordinate; they don't know about
  each other.
- **When to aggregate.** Aggregation is safe *after* the training
  run has called `run.finish()`, or at any time provided you only
  write `eval/*` keys (which the training script never writes).
  Never write training-owned keys from the aggregator.


## 8. Wrap up — what to look at in the W&B UI

**Group view.** Filter runs by `group=exp-XXXXXXXX` (see the
`experiment_id` printed in §1 / §3). One `train-*` run and three
`eval-*-epNNN` runs appear in one table.

**Lineage graph.** For each checkpoint version `model-exp-XXX:vN`:
- Output edge from the training run (produced it).
- Input edge into its eval run (via `use_artifact`).
- Each eval run also produces an `eval-results-exp-XXX:vN` artifact.

**Training run summary panel.** Now shows both training-owned keys
(`train/*`, `finished_at`, `train/checkpoint_refs`) and aggregated
eval keys (`eval/best/*`, `eval/latest/*`, `eval/history_count`,
`eval/run_ids`) — no collision because the eval keys are all
namespaced.
